In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

import copy
from torch.utils.data import DataLoader, TensorDataset

    
        
import matplotlib.pyplot as plt
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

import copy
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
from data_loading import *
from data_generation import *
from data_plot import *
from approaches.standardized_residuals import StandardizedResiduals
from models import *

In [ ]:
import matplotlib
import os
os.environ["PATH"] += os.pathsep + "/Library/TeX/texbin"

matplotlib.use('agg')
# matplotlib.use('pdf')
matplotlib.rcParams.update({
    "pgf.texsystem": "pdflatex",
    'font.family': 'serif',
    'font.size': 20,
    'text.usetex': True,
    'pgf.rcfonts': False,
    # 'legend.framealpha': 0.5,
    'text.latex.preamble': r'\usepackage{times} \usepackage{amsmath} \usepackage{amsfonts} \usepackage{amssymb} \usepackage{xcolor}'
})


In [ ]:
def generate_star(N, dim_y=2, loc=5.0, scale=2.0):
    if dim_y < 2:
        raise ValueError("Star shape requires at least 2 dimensions.")
    
    # Angles for the 5 outer points and 5 inner points of a star
    angles = np.linspace(0, 2 * np.pi, 11)
    r_outer = 2.0
    r_inner = 0.8
    
    radii = np.ones_like(angles)
    radii[::2] = r_outer
    radii[1::2] = r_inner
    
    pts_x = radii * np.cos(angles)
    pts_y = radii * np.sin(angles)
    
    # MODIFIED: Sample 't' from a Gaussian instead of linspace
    # loc=5.0 centers the cluster at the bottom point of the star
    t = np.random.normal(loc=loc, scale=scale, size=N)
    t = np.mod(t, 10.0) # Wrap around so points stay on the 10 edges
    
    star_x = np.interp(t, np.arange(11), pts_x)
    star_y = np.interp(t, np.arange(11), pts_y)
    
    star_data = np.stack([star_x, star_y], axis=1)
    star_tensor = torch.from_numpy(star_data).float()
    star_tensor += torch.randn_like(star_tensor) * 0.05 # Add jitter
    
    if dim_y > 2:
        extra = torch.zeros(N, dim_y - 2)
        star_tensor = torch.cat([star_tensor, extra], dim=1)
        
    return star_tensor

In [ ]:
torch.manual_seed(42)

N = 2000
dim_X = 1
dim_y = 2

# X est constant (identique pour tous)
X_train = torch.ones(N, dim_X)
Y_train = generate_star(N, dim_y)


X_calibration = X_train
Y_calibration = Y_train



tau = 0.70        # Cible de couverture : tau%
lambda_val = 1.
lr_lambda = 0.0
batch_size = 256
num_epochs = 1_000
lr = 5e-4


tau_param = TauParameterAnnealer(tau,                
                 warm_start_step=500, 
                 tau_low_target_step=100, 
                 tau_low_steepness=1e-3,
                 tau_high_target_step=100, 
                 tau_high_steepness=1e-2,
                 low_error_init=0.5,   
                 low_error_max=0.01,    
                 high_error_init=0.2,  
                 high_error_max=0.01,  
                 eps=1e-5              
                 )

model = UnifiedConditionalEstimator(dim_X=dim_X, dim_y=dim_y, 
                                    cov_mode="full_cholesky", num_flow_layers=6, K=1,
                                    det_normalized=True
                                    )


model.fit(X_train, Y_train, 
          tau=tau, 
          epochs=num_epochs, 
          lr=lr, 
          batch_size=batch_size, 
          return_best=True, 
          print_every=10,
          tau_parameterAnnealer=tau_param,
          loss_function="log_volume"
          )

model.conformalize(X_calibration, Y_calibration, tau, fake_for_trial=False)
vol = model.compute_average_volume(X_train[[0]])
print(f"Volume de la région de confiance: {vol.item():.4f}")

plot_1X(model, tau, X_train, Y_train)

In [ ]:
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def plot_1X(model, tau, X_train, Y_train, 
            res_points=150, 
            name=None, fontsize=16, 
            size_points=25, 
            color_set="#0C7C59", 
            color_points="#BAC1B8", 
            color_bound_points="#58A4B0", 
            xlim=None, 
            ylim=None,
            alpha_contour = 0.2
            ):
    model.eval()

    with torch.no_grad():
        # 1. Retrieve the calibrated threshold
        _, _, partition, q_conf = model.call_conformalize(X_train)
        q_val = q_conf[0, 0].item()
        
        # Calculate training scores
        S_y, _, _ = model.get_frontiers(X_train, Y_train)
        S_y = S_y.squeeze()
            
        inside_mask = (S_y <= q_val)
        empirical_coverage = inside_mask.float().mean().item()
        
        print(f"\nCouverture empirique : {empirical_coverage*100:.1f}% (Cible: {tau*100:.2f}%)")

        # 2. Grid creation
        if xlim is not None:
            y1_min, y1_max = xlim[0], xlim[1]
        else:
            y1_min, y1_max = Y_train[:, 0].min().item() - 1, Y_train[:, 0].max().item() + 1
        if xlim is not None:
            y2_min, y2_max = ylim[0], ylim[1]
        else:
            y2_min, y2_max = Y_train[:, 1].min().item() - 1, Y_train[:, 1].max().item() + 1
        
        y1_grid, y2_grid = torch.meshgrid(
            torch.linspace(y1_min, y1_max, res_points),
            torch.linspace(y2_min, y2_max, res_points),
            indexing='ij'
        )
        Y_grid = torch.stack([y1_grid.flatten(), y2_grid.flatten()], dim=1)
        
        # Repeat the first value of X_train for the conditional grid
        X_grid = X_train[0:1].repeat(Y_grid.shape[0], 1)
        
        # 3. Grid inference
        S_grid_raw, _, _ = model.get_frontiers(X_grid, Y_grid)
        S_grid = S_grid_raw.reshape(res_points, res_points)

    # 4. Plotting (NeurIPS Style)
    fig, ax = plt.subplots(figsize=(8, 5))


    # Shade the conformal region
    contour_fill = ax.contourf(y1_grid.numpy(), y2_grid.numpy(), S_grid.numpy(), 
                               levels=[-1e6, q_val], colors=[color_set], alpha=alpha_contour)

    # Draw the boundary of the conformal region
    contour_line = ax.contour(y1_grid.numpy(), y2_grid.numpy(), S_grid.numpy(), 
                              levels=[q_val], colors=[color_set], linewidths=2.5)

    # Plot all observations with a single color and white borders for clarity
    scatter = ax.scatter(Y_train[:, 0].numpy(), Y_train[:, 1].numpy(), 
                         color=color_points, alpha=0.5, s=size_points, 
                         edgecolors=color_bound_points, linewidths=0.5, zorder=0)

    # Create custom legend handles
    set_label = f"SLS"
    set_patch = mpatches.Patch(color=color_set, alpha=0.4, label=set_label)
    
    # We use a tuple for the scatter handle so it shows the marker properly
    scatter_handle = plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color_points, 
                                markeredgecolor=color_bound_points, markersize=8, alpha=0.75, label='Test points')

    # Formatting axes and text sizes
    ax.set_xlabel("$y_1$", fontsize=fontsize)
    ax.set_ylabel("$y_2$", fontsize=fontsize)
    
    # Increase tick label size
    ax.tick_params(axis='both', which='major', labelsize=fontsize - 2)

    # Apply detailed legend
    ax.legend(handles=[set_patch, scatter_handle], 
              fontsize=fontsize - 2, 
              loc='best', 
              framealpha=0.9, 
              edgecolor='lightgrey')

    # Subtle grid behind the data
    ax.grid(True, linestyle='--', alpha=0.4, zorder=0)
    
    # Set limits
    ax.set_xlim(y1_min, y1_max)
    ax.set_ylim(y2_min, y2_max)

    # Ensures no clipping of labels when saving to PDF
    plt.tight_layout() 
    if name is not None:
        plt.savefig(f"../figs/1D_plot_{name}_{empirical_coverage*100:.1f}.pdf", dpi=300)
    plt.show()

In [ ]:
plot_1X(model, tau, X_train, Y_train, res_points=1000, fontsize=20, name="star_V1", size_points=20, color_set="#FF5400", color_points="#FCFCFC", color_bound_points="#46237A")
plot_1X(model, tau, X_train, Y_train, res_points=1000, fontsize=20, alpha_contour=0.1, size_points=10, name="star_V2", color_set="#0C7C59", color_points="#BAC1B8", color_bound_points="#58A4B0")


In [ ]:
# plot_1X(model, tau, X_train, Y_train, res_points=1000, size_points=15, color_set="#0C7C59", color_points="#BAC1B8", color_bound_points="#58A4B0")

# plot_1X(model, tau, X_train, Y_train, res_points=1000, size_points=20, color_set="#FF5400", color_points="#FCFCFC", color_bound_points="#46237A")

# plot_1X(model, tau, X_train, Y_train, res_points=1000, size_points=20, color_set="#FF5400", color_points="#FCFCFC", color_bound_points="#256EFF")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def generate_data(n_samples_per_mode=1500):
    # Base distribution: Standard Normal z ~ N(0, I)
    # This guarantees the non-uniform density you requested.
    
    # ==========================================
    # 1. Ellipse Mode
    # ==========================================
    z1 = np.random.randn(n_samples_per_mode, 2)
    
    # Scale x strongly and y lightly, then rotate
    scale_matrix = np.array([[3.0, 0.0], 
                             [0.0, 0.5]])
    theta = np.pi / 4  # 45 degree rotation
    rotation_matrix = np.array([[np.cos(theta), -np.sin(theta)], 
                                [np.sin(theta), np.cos(theta)]])
    
    # Apply transformation and shift to top-left
    ellipse = z1 @ scale_matrix @ rotation_matrix + np.array([-6, 6])
    
    # ==========================================
    # 2. Banana Mode
    # ==========================================
    z2 = np.random.randn(n_samples_per_mode, 2)
    
    # Non-linear flow: stretch x, and bend y as a function of x^2
    banana_x = 2.5 * z2[:, 0]
    banana_y = 0.5 * z2[:, 1] + 0.5 * (z2[:, 0]**2 - 1)
    
    banana = np.column_stack((banana_x, banana_y))
    
    # Rotate slightly and shift to bottom-center
    theta_b = -np.pi / 6 # -30 degrees
    rot_b = np.array([[np.cos(theta_b), -np.sin(theta_b)], 
                      [np.sin(theta_b), np.cos(theta_b)]])
    banana = banana @ rot_b + np.array([0, -4])

    # ==========================================
    # 3. Strange Shape (Wavy "Snake" Blob)
    # ==========================================
    z3 = np.random.randn(n_samples_per_mode, 2)
    
    # Non-linear flow: stretch x massively, bend y with a sine wave
    strange_x = 2.0 * z3[:, 0]
    strange_y = 0.6 * z3[:, 1] + 1.2 * np.sin(1.5 * strange_x)
    
    strange = np.column_stack((strange_x, strange_y))
    
    # Shift to top-right
    strange = strange + np.array([6, 5])

    data = np.concatenate([strange, banana, ellipse])
    data = data /10

    return torch.tensor(data, dtype=torch.float32)


torch.manual_seed(42)


N = 3000
dim_X = 1
dim_y = 2

X_train = torch.ones(N, dim_X)
Y_train = generate_data(n_samples_per_mode=N//3)


X_calibration = X_train
Y_calibration = Y_train


tau = 0.90       # Cible de couverture : tau%

batch_size = 256
num_epochs = 1_000
lr = 5e-4


tau_param = TauParameterAnnealer(tau,                
                 warm_start_step=500, 
                 tau_low_target_step=100, 
                 tau_low_steepness=1e-3,
                 tau_high_target_step=100, 
                 tau_high_steepness=1e-2,
                 low_error_init=0.5,   
                 low_error_max=0.01,    
                 high_error_init=0.2,  
                 high_error_max=0.01,  
                 eps=1e-5              
                 )

model = UnifiedConditionalEstimator(dim_X=dim_X, dim_y=dim_y, 
                                    cov_mode="full_cholesky", num_flow_layers=6, K=4,
                                    det_normalized=True
                                    )


model.fit(X_train, Y_train, 
          tau=tau, 
          epochs=num_epochs, 
          lr=lr, 
          batch_size=batch_size, 
          return_best=True, 
          print_every=10,
          tau_parameterAnnealer=tau_param,
          loss_function="log_volume"
          )

model.conformalize(X_calibration, Y_calibration, tau, fake_for_trial=False)
vol = model.compute_average_volume(X_train[[0]])
print(f"Volume de la région de confiance: {vol.item():.4f}")

plot_1X(model, tau, X_train, Y_train)


In [ ]:
plot_1X(model, 
        tau, 
        X_train, 
        Y_train, 
        res_points=1000, 
        fontsize=20, 
        name="multiple_V1", 
        size_points=20, 
        color_set="#FF5400", 
        color_points="#FCFCFC", 
        color_bound_points="#46237A",
        xlim=(-1.5, 1.5),
        ylim=(-0.7, 1.5)
        )

plot_1X(model, 
        tau, 
        X_train, 
        Y_train, 
        res_points=1000, 
        fontsize=20, 
        alpha_contour=0.1,
        name="multiple_V2", 
        size_points=20, 
        color_set="#0C7C59",
        color_points="#BAC1B8",
        color_bound_points="#58A4B0",
        xlim=(-1.5, 1.5),
        ylim=(-0.7, 1.5)
        )


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def plot_1d_conditional_contours(
    data_generator,
    model,
    tau=0.90,
    n_samples=50,
    res_points=100,
    plot_mu=True,
    f=None,
    name=None, 
    fontsize=16, 
    size_points=25, 
    color_set="#0C7C59", 
    color_points="#BAC1B8", 
    color_bound_points="#58A4B0", 
    alpha_fill=0.2,
    xlim=None, 
    ylim=None
):

    model.eval()

    # Gestion des limites X
    if xlim is not None:
        x_min, x_max = xlim[0], xlim[1]
    else:
        x_min, x_max = -1.0, 1.0

    xs = torch.linspace(x_min, x_max, 500)

    # --------------------------------------------------
    # 1. Génération des samples Y|X pour visualisation
    # --------------------------------------------------

    y_all = []
    samples_all = []
    x_samples_all = []

    for x_val in xs:

        x_tensor = x_val.view(1, 1)

        samples = data_generator.generate_specific_y_given_x(
            x_tensor,
            n=n_samples
        )

        y_all.append(samples)
        samples_all.append(samples)
        x_samples_all.append(torch.full((n_samples, 1), x_val))

    samples_all = torch.cat(samples_all).numpy()
    x_samples_all = torch.cat(x_samples_all).numpy()
    y_all = torch.cat(y_all, dim=0)

    # Gestion des limites Y
    if ylim is not None:
        y_min, y_max = ylim[0], ylim[1]
    else:
        y_min = y_all.min().item() - 1
        y_max = y_all.max().item() + 1

    # --------------------------------------------------
    # 2. Construction de la grille (X,Y)
    # --------------------------------------------------

    y_grid_vals = torch.linspace(y_min, y_max, res_points)
    Xg, Yg = torch.meshgrid(xs, y_grid_vals, indexing="ij")

    X_grid = Xg.reshape(-1, 1)
    Y_grid = Yg.reshape(-1, 1)

    # --------------------------------------------------
    # 3. Calcul du score S(x,y) et de la couverture
    # --------------------------------------------------

    with torch.no_grad():
        S_grid_raw, _, _ = model.get_frontiers(X_grid, Y_grid)
        S_grid = S_grid_raw.reshape(len(xs), res_points)

        conformalize_out = model.call_conformalize(xs.unsqueeze(1))
        q_vals = conformalize_out[-1][:, 0]
        
        # Calcul de la couverture empirique sur les échantillons générés
        X_samp_tensor = torch.tensor(x_samples_all, dtype=torch.float32)
        Y_samp_tensor = torch.tensor(samples_all, dtype=torch.float32)

        S_samp, _, _ = model.get_frontiers(X_samp_tensor, Y_samp_tensor)
        q_samp = model.call_conformalize(X_samp_tensor)[-1][:, 0]
        
        inside_mask = (S_samp.squeeze() <= q_samp.squeeze())
        empirical_coverage = inside_mask.float().mean().item()
        
        print(f"\nCouverture empirique : {empirical_coverage*100:.1f}% (Cible: {tau*100:.2f}%)")

    q_grid = q_vals.unsqueeze(1).repeat(1, res_points)
    
    # On calcule S - q pour que la frontière soit exactement au niveau 0
    S_diff = (S_grid - q_grid).numpy()

    # --------------------------------------------------
    # 4. Plot
    # --------------------------------------------------

    fig, ax = plt.subplots(figsize=(8, 5))

    X_plot, Y_plot = np.meshgrid(xs.numpy(), y_grid_vals.numpy(), indexing="ij")

    # Ombrage de la région conforme
    ax.contourf(
        X_plot, Y_plot, S_diff,
        levels=[-1e6, 0], colors=[color_set], alpha=alpha_fill
    )

    # Frontière de la région
    ax.contour(
        X_plot, Y_plot, S_diff,
        levels=[0], colors=[color_set], linewidths=2.5
    )

    # Points de test
    ax.scatter(
        x_samples_all, samples_all,
        color=color_points, alpha=0.5, s=size_points, 
        edgecolors=color_bound_points, linewidths=0.5, zorder=0
    )

    # Création des handles pour la légende
    handles = []
    set_patch = mpatches.Patch(color=color_set, alpha=0.4, label="SLS")
    handles.append(set_patch)

    scatter_handle = plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color_points, 
                                markeredgecolor=color_bound_points, markersize=8, alpha=0.75, label='Test points')
    handles.append(scatter_handle)

    # --------------------------------------------------
    # 5. Courbes mu_k(x) et f(x)
    # --------------------------------------------------

    if plot_mu:
        with torch.no_grad():
            mu_vals = conformalize_out[0]

        for k in range(model.K):
            ax.plot(
                xs.numpy(), mu_vals[:, k, 0].numpy(),
                linewidth=2, color="#D62828", zorder=2 # Rouge brique pour trancher avec la nouvelle palette
            )
            
        mu_handle = plt.Line2D([0], [0], color="#D62828", linewidth=2, label='$\mu_k(x)$')
        handles.append(mu_handle)

    if f is not None:
        mu_vals_f = f(xs.unsqueeze(1))
        ax.plot(
            xs.numpy(), mu_vals_f[:, 0].numpy(),
            linewidth=2, color="#F77F00", zorder=2 # Orange foncé pour f(x)
        )
        f_handle = plt.Line2D([0], [0], color="#F77F00", linewidth=2, label='$f(x)$')
        handles.append(f_handle)

    # --------------------------------------------------
    # 6. Mise en forme des axes
    # --------------------------------------------------

    ax.set_xlabel("$x$", fontsize=fontsize)
    ax.set_ylabel("$y$", fontsize=fontsize)
    
    # Taille des ticks
    ax.tick_params(axis='both', which='major', labelsize=fontsize - 2)

    # Application de la légende
    ax.legend(
        handles=handles, 
        fontsize=fontsize - 2, 
        loc='best', 
        framealpha=0.9, 
        edgecolor='lightgrey'
    )

    # Grille subtile
    ax.grid(True, linestyle='--', alpha=0.4, zorder=0)

    # Application des limites
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)

    plt.tight_layout()
    
    # Sauvegarde (avec un suffixe _cond_ pour ne pas écraser les plots de l'autre fonction)
    if name is not None:
        plt.savefig(f"../figs/1D_cond_plot_{name}_{empirical_coverage*100:.1f}.pdf", dpi=300)
        
    plt.show()


In [ ]:
import torch

class GeneratorD:
    def __init__(self, f, matrix_transform, dim_y, noise_type='gaussian', noise_std=1.0, transition_center=0.0, transition_steepness=5.0):
        self.f = f
        self.matrix_transform = matrix_transform
        self.noise_type = noise_type
        self.noise_std = noise_std
        self.dim_y = dim_y
        
        # Parameters for the smooth transition
        self.transition_center = transition_center
        self.transition_steepness = transition_steepness

    def _get_noise(self, n, x=None):
        if self.noise_type == 'gaussian':
            noise = torch.randn(n, self.dim_y)
        elif self.noise_type == 'uniform':
            noise = torch.rand(n, self.dim_y) * 2 - 1
        elif self.noise_type == 'exponential':
            noise = torch.distributions.Exponential(rate=1.0).sample((n, self.dim_y)) - 1.0
        elif self.noise_type == 'multimodal':
            noise1 = torch.distributions.Exponential(rate=1.0).sample((n, self.dim_y)) - 1.0
            noise2 = - torch.distributions.Exponential(rate=1.0).sample((n, self.dim_y)) - 3.0
            mask = (torch.rand(n, 1) > 0.5).float()
            noise = mask * noise1 + (1.0 - mask) * noise2
            
        elif self.noise_type == 'smooth_bilevel':
            if x is None:
                raise ValueError("x must be provided to generate state-dependent 'smooth_bilevel' noise.")
            
            # 1. Unimodal Noise (Gaussian)
            unimodal_noise = torch.randn(n, self.dim_y)
            
            # 2. Bimodal Noise (Two opposing exponentials)
            exp1 = torch.distributions.Exponential(rate=1.0).sample((n, self.dim_y)) - 1.0
            exp2 = -torch.distributions.Exponential(rate=1.0).sample((n, self.dim_y)) - 3.0
            mask_bimodal = (torch.rand(n, 1) > 0.5).float()
            bimodal_noise = mask_bimodal * exp1 + (1.0 - mask_bimodal) * exp2
            
            # 3. Smooth Sigmoid Weighting based on X
            # alpha approaches 0 for low X, and 1 for large X
            alpha = torch.sigmoid(self.transition_steepness * (x - self.transition_center))
            
            # Ensure broadcast compatibility if dim_y > 1
            if alpha.shape[1] == 1 and self.dim_y > 1:
                alpha = alpha.expand(n, self.dim_y)
                
            # 4. Interpolate between the two distributions
            noise = (1.0 - alpha) * unimodal_noise + alpha * bimodal_noise
            
        else:
            raise ValueError(f"Type inconnu: {self.noise_type}")
            
        return (noise * self.noise_std).unsqueeze(2)

    def generate(self, n):
        x = 2 * torch.rand(n, 1) - 1
        fx = self.f(x)
        A_x = self.matrix_transform(x)
        
        noise = self._get_noise(n, x=x)
        
        correlated_noise = torch.bmm(A_x, noise).squeeze(2)
        y = fx + correlated_noise
        return x, y

    def generate_specific_y_given_x(self, x_tensor, n=1):
        x_repeated = x_tensor.repeat_interleave(n, dim=0)
        fx = self.f(x_repeated)
        A_x = self.matrix_transform(x_repeated)
        
        noise = self._get_noise(x_repeated.shape[0], x=x_repeated)
        
        correlated_noise = torch.bmm(A_x, noise).squeeze(2)
        y_flat = fx + correlated_noise
        return y_flat.view(n, self.dim_y)

def strange_matrix_transform_1D(x):
    n = x.shape[0]
    matrices = torch.eye(1).unsqueeze(0).repeat(n, 1, 1)
    matrices[:, 0, 0] = x.squeeze(-1)**2 + 0.5 
    return matrices

def circle_f_1D(x):
    return torch.sin(x*3)


# ==========================================
# 3. Entraînement
# ==========================================
torch.manual_seed(42)

generator = GeneratorD(
    f=circle_f_1D,
    matrix_transform=strange_matrix_transform_1D,
    dim_y=1,
    noise_std=1.0,
    noise_type="smooth_bilevel"
)

# On génère un dataset d'entraînement
N_train = 3000
X_train, Y_train = generator.generate(N_train)
X_val, Y_val = generator.generate(N_train)
X_calibration, Y_calibration = generator.generate(N_train)
X_test, Y_test = generator.generate(N_train)

dim_X = 1
dim_y = 1


In [ ]:
torch.manual_seed(42)
tau = 0.80      

batch_size = 256
num_epochs = 250
lr = 5e-4


tau_param = TauParameterAnnealer(tau,                
                 warm_start_step=500, 
                 tau_low_target_step=100, 
                 tau_low_steepness=1e-3,
                 tau_high_target_step=100, 
                 tau_high_steepness=1e-2,
                 low_error_init=0.5,   
                 low_error_max=0.01,    
                 high_error_init=0.2,  
                 high_error_max=0.01,  
                 eps=1e-5              
                 )


model = UnifiedConditionalEstimator(dim_X=dim_X, dim_y=dim_y, 
                                    cov_mode="full_cholesky", num_flow_layers=3, K=3,
                                    det_normalized=True
                                    )


model.fit(X_train, Y_train, 
          tau=tau, 
          epochs=num_epochs, 
          lr=lr, 
          batch_size=batch_size, 
          return_best=True, 
          print_every=10,
          tau_parameterAnnealer=tau_param,
          loss_function="log_volume"
          )

model.conformalize(X_calibration, Y_calibration, tau, fake_for_trial=False)
vol = model.compute_average_volume(X_test)
print(f"Volume de la région de confiance: {vol.item():.4f}")

plot_1d_conditional_contours(generator, model, f=circle_f_1D, plot_mu=False)

In [ ]:

plot_1d_conditional_contours(generator, 
                             model, 
                             plot_mu=False,
                             fontsize=20, 
                             n_samples = 30,
                             name="bimodal_V1", 
                             size_points=10, 
                             color_set="#FF5400", 
                             color_points="#FCFCFC", 
                             color_bound_points="#46237A",
                             )

plot_1d_conditional_contours(generator, 
                             model, 
                             plot_mu=False,
                             fontsize=20, 
                             n_samples = 30,
                             alpha_fill = 0.1,
                             name="bimodal_V2", 
                             size_points=10, 
                             color_set="#0C7C59",
                             color_points="#BAC1B8",
                             color_bound_points="#58A4B0",
                             )


In [ ]:
torch.manual_seed(42)

generator = GeneratorD(
    f=circle_f_1D,
    matrix_transform=strange_matrix_transform_1D,
    dim_y=1,
    noise_std=1.0,
    noise_type="exponential"
)

# On génère un dataset d'entraînement
N_train = 3000
X_train, Y_train = generator.generate(N_train)
X_val, Y_val = generator.generate(N_train)
X_calibration, Y_calibration = generator.generate(N_train)
X_test, Y_test = generator.generate(N_train)

dim_X = 1
dim_y = 1

In [ ]:
torch.manual_seed(42)

tau = 0.6

batch_size = 256
num_epochs = 1_000
lr = 5e-4


tau_param = TauParameterAnnealer(tau,                
                 warm_start_step=5, 
                 tau_low_target_step=100, 
                 tau_low_steepness=1e-3,
                 tau_high_target_step=100, 
                 tau_high_steepness=1e-2,
                 low_error_init=0.5,   
                 low_error_max=0.01,    
                 high_error_init=0.2,  
                 high_error_max=0.01,  
                 eps=1e-5              
                 )


model = UnifiedConditionalEstimator(dim_X=dim_X, dim_y=dim_y, 
                                    cov_mode="full_cholesky", num_flow_layers=0, K=1,
                                    det_normalized=True
                                    )


model.fit(X_train, Y_train, 
          tau=tau, 
          epochs=num_epochs, 
          lr=lr, 
          batch_size=batch_size, 
          return_best=True, 
          print_every=10,
          tau_parameterAnnealer=tau_param,
          loss_function="log_volume"
          )

model.conformalize(X_calibration, Y_calibration, tau, fake_for_trial=False)
vol = model.compute_average_volume(X_test)
print(f"Volume de la région de confiance: {vol.item():.4f}")

plot_1d_conditional_contours(generator, model, f=circle_f_1D, plot_mu=False)

In [ ]:

plot_1d_conditional_contours(generator, 
                             model, 
                             plot_mu=False,
                             fontsize=20, 
                             name="exponential_V1", 
                             n_samples = 30,
                             size_points=10, 
                             color_set="#FF5400", 
                             color_points="#FCFCFC", 
                             color_bound_points="#46237A",
                             )

plot_1d_conditional_contours(generator, 
                             model, 
                             plot_mu=False,
                             fontsize=20, 
                             alpha_fill=0.1,
                             n_samples = 30,
                             name="exponential_V2", 
                             size_points=10, 
                             color_set="#0C7C59",
                             color_points="#BAC1B8",
                             color_bound_points="#58A4B0",
                             )


In [ ]:


import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def plot_samples_with_contours_multi_flow(
    data_generator, 
    model, 
    add_outliers=None,
    tau=0.9, 
    x_range=(-1, 1), 
    num_slices=8, 
    n_samples=200,
    name=None, 
    fontsize=16, 
    size_points=15, 
    color_set="#0C7C59", 
    color_points="#BAC1B8", 
    color_bound_points="#58A4B0", 
    ylim=None, 
    zlim=None
):
    model.eval()

    fig = plt.figure(figsize=(10, 7), dpi=120)
    ax = fig.add_subplot(111, projection='3d')

    xs = np.linspace(x_range[0], x_range[1], num_slices)

    for x_val in xs:

        x_tensor = torch.tensor([[x_val]], dtype=torch.float32)
        samples = data_generator.generate_specific_y_given_x(x_tensor, n=n_samples)
        if add_outliers is not None:
            samples = add_outliers(samples)
        samples_np = samples.numpy()

        y1_min, y1_max = samples[:, 0].min().item() - 1, samples[:, 0].max().item() + 1
        y2_min, y2_max = samples[:, 1].min().item() - 1, samples[:, 1].max().item() + 1

        y1_grid_vals = np.linspace(y1_min, y1_max, 100)
        y2_grid_vals = np.linspace(y2_min, y2_max, 100)

        Y1, Y2 = np.meshgrid(y1_grid_vals, y2_grid_vals)

        y1_tensor = torch.tensor(Y1.flatten(), dtype=torch.float32)
        y2_tensor = torch.tensor(Y2.flatten(), dtype=torch.float32)

        Y_grid = torch.stack([y1_tensor, y2_tensor], dim=1)

        x_scatter = np.full(n_samples, x_val)

        # Plot des points de test avec la nouvelle stylisation
        ax.scatter(
            x_scatter,
            samples_np[:, 0],
            samples_np[:, 1],
            color=color_points,
            s=size_points,
            alpha=0.6,
            edgecolors=color_bound_points,
            linewidths=0.5,
            zorder=1
        )

        with torch.no_grad():
            # Récupération du seuil calibré q_val
            conformalize_out = model.call_conformalize(x_tensor)
            q_val = conformalize_out[-1][0, 0].item()

            # Calcul des scores de la grille
            X_grid = x_tensor.repeat(Y_grid.shape[0], 1)
            S_grid_raw, _, _ = model.get_frontiers(X_grid, Y_grid)
            
            # Redimensionnement du score brut pour l'affichage
            S_grid_2d = S_grid_raw.reshape(Y1.shape).numpy()

        if S_grid_2d.min() <= q_val <= S_grid_2d.max():

            fig_dummy, ax_dummy = plt.subplots()

            cs = ax_dummy.contour(Y1, Y2, S_grid_2d, levels=[q_val])

            paths = cs.collections[0].get_paths() if hasattr(cs, 'collections') else cs.get_paths()

            for path in paths:

                vertices = path.vertices

                x_line = np.full(vertices.shape[0], x_val)
                y1_line = vertices[:, 0]
                y2_line = vertices[:, 1]

                # Lignes de contour avec color_set
                ax.plot(
                    x_line,
                    y1_line,
                    y2_line,
                    color=color_set,
                    linewidth=2.5,
                    zorder=10
                )

            plt.close(fig_dummy)

        else:
            print(f"Attention: quantile {q_val:.2f} hors grille pour x={x_val:.2f}")

    # Création d'une légende cohérente
    set_handle = plt.Line2D([0], [0], color=color_set, linewidth=2.5, label=f"SLS")
    scatter_handle = plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color_points, 
                                markeredgecolor=color_bound_points, markersize=8, alpha=0.75, label='Test points')
    
    ax.legend(handles=[set_handle, scatter_handle], 
              fontsize=fontsize - 2, 
              loc='upper left', 
              framealpha=0.9, 
              edgecolor='lightgrey')

    # Mise en forme des axes
    ax.set_xlabel('$x$', fontsize=fontsize, labelpad=10)
    ax.set_ylabel('$y_1$', fontsize=fontsize, labelpad=10)
    ax.set_zlabel('$y_2$', fontsize=fontsize, labelpad=10)

    # Application des limites
    ax.set_xlim(x_range[0], x_range[1])
    
    if ylim is not None:
        ax.set_ylim(ylim[0], ylim[1])
    else:
        ax.set_ylim(-4, 4)
        
    if zlim is not None:
        ax.set_zlim(zlim[0], zlim[1])
    else:
        ax.set_zlim(-2, 6)

    # Nettoyage visuel des panneaux 3D
    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False

    # Taille des labels de ticks
    ax.tick_params(axis='both', which='major', labelsize=fontsize - 4)

    ax.view_init(elev=20, azim=-67)

    plt.tight_layout()
    
    # Sauvegarde
    if name is not None:
        plt.savefig(f"../figs/3D_flow_plot_{name}_{tau*100:.0f}.pdf", dpi=300, bbox_inches='tight')
        
    plt.show()

In [ ]:
torch.manual_seed(42)

generator = Generator2D(
    f=circle_f,
    matrix_transform=strange_matrix_transform,
    noise_std=1.0,
    noise_type="exponential"
)

# On génère un dataset d'entraînement
N_train = 3000
X_train, Y_train = generator.generate(N_train)
X_calibration, Y_calibration = generator.generate(N_train)
X_test, Y_test = generator.generate(N_train)

dim_X = 1
dim_y = 2

tau = 0.8

batch_size = 256
num_epochs = 1_000
lr = 5e-4

tau_param = TauParameterAnnealer(tau,                
                 warm_start_step=500, 
                 tau_low_target_step=100, 
                 tau_low_steepness=1e-3,
                 tau_high_target_step=100, 
                 tau_high_steepness=1e-2,
                 low_error_init=0.5,   
                 low_error_max=0.01,    
                 high_error_init=0.2,  
                 high_error_max=0.01,  
                 eps=1e-5              
                 )

model = UnifiedConditionalEstimator(dim_X=dim_X, dim_y=dim_y, 
                                    cov_mode="full_cholesky", num_flow_layers=6, K=1,
                                    det_normalized=True
                                    )


model.fit(X_train, Y_train, 
          tau=tau, 
          epochs=num_epochs, 
          lr=lr, 
          batch_size=batch_size, 
          return_best=True, 
          print_every=10,
          tau_parameterAnnealer=tau_param,
          loss_function="log_volume"
          )

model.conformalize(X_calibration, Y_calibration, tau, fake_for_trial=False)
vol = model.compute_average_volume(X_test)
print(f"Volume de la région de confiance: {vol.item():.4f}")



plot_samples_with_contours_multi_flow(generator, model, tau=tau, num_slices=10)

In [ ]:
# plot_samples_with_contours_multi_flow(generator, model, tau=tau, num_slices=10)


plot_samples_with_contours_multi_flow(generator, 
                             model, 
                             fontsize=20, 
                             name="2D_exponential", 
                             size_points=10, 
                             color_set="#FF5400", 
                             color_points="#FCFCFC", 
                             color_bound_points="#46237A",
                             num_slices=10,
                             )

plot_samples_with_contours_multi_flow(generator, 
                             model, 
                             fontsize=20, 
                             name="2D_exponential_V2", 
                             size_points=10, 
                             color_set="#0C7C59",
                             color_points="#BAC1B8",
                             color_bound_points="#58A4B0",
                             num_slices=10,
                             )


In [ ]:
import torch

class Generator2D:
    def __init__(self, f, matrix_transform, noise_type='gaussian', noise_std=1.0):
        self.f = f
        self.matrix_transform = matrix_transform
        self.noise_type = noise_type
        self.noise_std = noise_std

    def _get_noise(self, n):
        if self.noise_type == 'gaussian':
            noise = torch.randn(n, 2)
        elif self.noise_type == 'uniform':
            noise = torch.rand(n, 2) * 2 - 1
        elif self.noise_type == 'exponential':
            noise = torch.distributions.Exponential(rate=1.0).sample((n, 2))
        elif self.noise_type == 'outliers':
            # 1. Base Gaussian noise (center 0, std 1)
            base_noise = torch.randn(n, 2)
            
            # 2. Outlier noise
            # Reduced variance (std=3.0) and shifted to a very different center (e.g., X=10, Y=-10)
            outlier_center = torch.tensor([[0, 1.0]])
            outlier_noise = torch.randn(n, 2) * 1.0 + outlier_center
            
            # 3. Mask for 10% probability of being an outlier
            mask = (torch.rand(n, 1) < 0.10).float()
            mask = mask.expand(n, 2)
            
            # 4. Combine based on the mask
            noise = (1.0 - mask) * base_noise + mask * outlier_noise
        else:
            raise ValueError(f"Type inconnu: {self.noise_type}")
            
        return (noise * self.noise_std).unsqueeze(2)

    def generate(self, n):
        x = 2 * torch.rand(n, 1) - 1
        fx = self.f(x)
        A_x = self.matrix_transform(x)
        noise = self._get_noise(n)
        correlated_noise = torch.bmm(A_x, noise).squeeze(2)
        y = fx + correlated_noise
        return x, y

    def generate_specific_y_given_x(self, x_tensor, n=1):
        x_repeated = x_tensor.repeat_interleave(n, dim=0)
        fx = self.f(x_repeated)
        A_x = self.matrix_transform(x_repeated)
        noise = self._get_noise(x_repeated.shape[0])
        correlated_noise = torch.bmm(A_x, noise).squeeze(2)
        y_flat = fx + correlated_noise
        return y_flat.view(n, 2)

def strange_matrix_transform(x):
    n = x.shape[0]
    matrices = torch.eye(2).unsqueeze(0).repeat(n, 1, 1)
    matrices[:, 0, 0] = x.squeeze(-1)**2 + 0.5 
    matrices[:, 0, 1] = torch.sin(x.squeeze(-1) * 2)
    matrices[:, 1, 1] = torch.abs(x.squeeze(-1)) + 0.2
    return matrices

def circle_f(x):
    return torch.cat([torch.sin(x*3), torch.cos(x*3)], dim=1)

In [ ]:
def add_outliers(Y):
    n = len(Y)
    outlier_center = torch.tensor([[3.0, 4.0]])
    outlier_noise = torch.randn(n, 2) * 6.0 + outlier_center
    
    # 3. Mask for 10% probability of being an outlier
    mask = (torch.rand(n, 1) < 0.10).float()
    mask = mask.expand(n, 2)
    
    # 4. Combine based on the mask
    noise = (1.0 - mask) * Y + mask * outlier_noise
    return noise

import torch

def add_outliers(Y):
    n = len(Y)
    outlier_center = torch.tensor([[3.0, 4.0]])
    
    outlier_noise = 4 * torch.rand(n, 2) - 2 + outlier_center
    
    # 3. Mask for 10% probability of being an outlier
    mask = (torch.rand(n, 1) < 0.10).float()
    mask = mask.expand(n, 2)
    
    # 4. Combine based on the mask
    noise = (1.0 - mask) * Y + mask * outlier_noise
    return noise

In [ ]:
torch.manual_seed(42)

generator = Generator2D(
    f=circle_f,
    matrix_transform=strange_matrix_transform,
    noise_std=1.0,
    noise_type="gaussian"
)

# On génère un dataset d'entraînement
N_train = 3000
X_train, Y_train = generator.generate(N_train)
X_calibration, Y_calibration = generator.generate(N_train)
X_test, Y_test = generator.generate(N_train)

Y_train = add_outliers(Y_train)
Y_calibration = add_outliers(Y_calibration)
Y_test = add_outliers(Y_test)

dim_X = 1
dim_y = 2

tau = 0.6

batch_size = 256
num_epochs = 1_000
lr = 5e-4

tau_param = TauParameterAnnealer(tau,                
                 warm_start_step=500, 
                 tau_low_target_step=100, 
                 tau_low_steepness=1e-3,
                 tau_high_target_step=100, 
                 tau_high_steepness=1e-2,
                 low_error_init=0.5,   
                 low_error_max=0.01,    
                 high_error_init=0.2,  
                 high_error_max=0.01,  
                 eps=1e-5              
                 )

model = UnifiedConditionalEstimator(dim_X=dim_X, dim_y=dim_y, 
                                    cov_mode="full_cholesky", num_flow_layers=0, K=1,
                                    det_normalized=True
                                    )


model.fit(X_train, Y_train, 
          tau=tau, 
          epochs=num_epochs, 
          lr=lr, 
          batch_size=batch_size, 
          return_best=True, 
          print_every=10,
          tau_parameterAnnealer=tau_param,
          loss_function="log_volume"
          )

model.conformalize(X_calibration, Y_calibration, tau, fake_for_trial=False)
vol = model.compute_average_volume(X_test)
print(f"Volume de la région de confiance: {vol.item():.4f}")



plot_samples_with_contours_multi_flow(generator, model, tau=tau, add_outliers=add_outliers, num_slices=10)

In [ ]:
# plot_samples_with_contours_multi_flow(generator, model, tau=tau, num_slices=10)


plot_samples_with_contours_multi_flow(generator, 
                             model, 
                             fontsize=20, 
                             add_outliers=add_outliers,
                             name="2D_gaussian_outliers", 
                             size_points=10, 
                             color_set="#FF5400", 
                             color_points="#FCFCFC", 
                             color_bound_points="#46237A",
                             num_slices=10,
                             )

plot_samples_with_contours_multi_flow(generator, 
                             model, 
                             fontsize=20, 
                             add_outliers=add_outliers,
                             name="2D_gaussian_outliers_V2", 
                             size_points=10, 
                             color_set="#0C7C59",
                             color_points="#BAC1B8",
                             color_bound_points="#58A4B0",
                             num_slices=10,
                             )


In [ ]:
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def plot_with_gaussian(model, gaussian_model, tau, X_train, Y_train, 
            res_points=150, 
            name=None, fontsize=16, 
            size_points=25, 
            color_set="#0C7C59", 
            color_gaussian="#350C7C", 
            color_points="#BAC1B8", 
            color_bound_points="#58A4B0", 
            xlim=None, 
            ylim=None,
            alpha_contour = 0.2
            ):
    model.eval()

    with torch.no_grad():
        # 1. Retrieve the calibrated threshold
        _, _, partition, q_conf = model.call_conformalize(X_train)
        q_val = q_conf[0, 0].item()
        
        # Calculate training scores
        S_y, _, _ = model.get_frontiers(X_train, Y_train)
        S_y = S_y.squeeze()
            
        inside_mask = (S_y <= q_val)
        empirical_coverage = inside_mask.float().mean().item()

        # --- PARTIE GAUSSIENNE ---
        with torch.no_grad():
            center, sigma = gaussian_model.get_distribution(X_train[[0]])
            
            # Décomposition pour l'ellipse
            L_eig, Q_eig = torch.linalg.eigh(sigma)
            L_sqrt = torch.diag_embed(torch.sqrt(L_eig.clamp(min=1e-12)))
            sigma_sqrt = (Q_eig @ L_sqrt @ Q_eig.transpose(-2, -1)).squeeze(0).cpu().numpy()
            mu_gauss = center.squeeze(0).cpu().numpy()
            
            theta = np.linspace(0, 2*np.pi, 100)
            circle = np.stack([np.cos(theta), np.sin(theta)], axis=0)
            ellipse_pts = mu_gauss[:, None] + sigma_sqrt @ (gaussian_model.q_alpha * circle)
        
        print(f"\nCouverture empirique : {empirical_coverage*100:.1f}% (Cible: {tau*100:.2f}%)")

        # 2. Grid creation
        if xlim is not None:
            y1_min, y1_max = xlim[0], xlim[1]
        else:
            y1_min, y1_max = Y_train[:, 0].min().item() - 1, Y_train[:, 0].max().item() + 1
        if xlim is not None:
            y2_min, y2_max = ylim[0], ylim[1]
        else:
            y2_min, y2_max = Y_train[:, 1].min().item() - 1, Y_train[:, 1].max().item() + 1
        
        y1_grid, y2_grid = torch.meshgrid(
            torch.linspace(y1_min, y1_max, res_points),
            torch.linspace(y2_min, y2_max, res_points),
            indexing='ij'
        )
        Y_grid = torch.stack([y1_grid.flatten(), y2_grid.flatten()], dim=1)
        
        # Repeat the first value of X_train for the conditional grid
        X_grid = X_train[0:1].repeat(Y_grid.shape[0], 1)
        
        # 3. Grid inference
        S_grid_raw, _, _ = model.get_frontiers(X_grid, Y_grid)
        S_grid = S_grid_raw.reshape(res_points, res_points)

    # 4. Plotting (NeurIPS Style)
    fig, ax = plt.subplots(figsize=(8, 6))


    # Shade the conformal region
    contour_fill = ax.contourf(y1_grid.numpy(), y2_grid.numpy(), S_grid.numpy(), 
                               levels=[-1e6, q_val], colors=[color_set], alpha=alpha_contour)
    
    gaussian_handle, = ax.plot(ellipse_pts[0, :], ellipse_pts[1, :], 
                           color=color_gaussian, linestyle='--', 
                           linewidth=2, label='Gaussian')

    # Draw the boundary of the conformal region
    contour_line = ax.contour(y1_grid.numpy(), y2_grid.numpy(), S_grid.numpy(), 
                              levels=[q_val], colors=[color_set], linewidths=2.5)

    # Plot all observations with a single color and white borders for clarity
    scatter = ax.scatter(Y_train[:, 0].numpy(), Y_train[:, 1].numpy(), 
                         color=color_points, alpha=0.5, s=size_points, 
                         edgecolors=color_bound_points, linewidths=0.5, zorder=0)

    # Create custom legend handles
    set_label = f"SLS"
    set_patch = mpatches.Patch(color=color_set, alpha=0.4, label=set_label)
    
    # We use a tuple for the scatter handle so it shows the marker properly
    scatter_handle = plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color_points, 
                                markeredgecolor=color_bound_points, markersize=8, alpha=0.75, label='Test points')

    # Formatting axes and text sizes
    ax.set_xlabel("$y_1$", fontsize=fontsize)
    ax.set_ylabel("$y_2$", fontsize=fontsize)
    
    # Increase tick label size
    ax.tick_params(axis='both', which='major', labelsize=fontsize - 2)

    # Apply detailed legend
    ax.legend(handles=[gaussian_handle, set_patch, scatter_handle], 
              fontsize=fontsize - 2, 
              loc='best', 
              framealpha=0.9, 
              edgecolor='lightgrey')

    # Subtle grid behind the data
    ax.grid(True, linestyle='--', alpha=0.4, zorder=0)
    
    # Set limits
    ax.set_xlim(y1_min, y1_max)
    ax.set_ylim(y2_min, y2_max)

    # Ensures no clipping of labels when saving to PDF
    plt.tight_layout() 
    if name is not None:
        plt.savefig(f"../figs/1D_plot_{name}_{empirical_coverage*100:.1f}.pdf", dpi=300)
    plt.show()

In [ ]:
torch.manual_seed(42)

N = 4000
dim_X = 1
dim_y = 2

# X est constant (identique pour tous)
X_train = torch.ones(N, dim_X)

# Y suit une loi exponentielle 2D indépendante
outlier_fraction = 0.05
num_outliers = int(N * outlier_fraction)

Y_train = torch.empty(N, dim_y).normal_(mean=1.0, std=1.0)
# outliers = torch.empty(num_outliers, dim_y).normal_(mean=10.0, std=5.0)
outliers = torch.rand(num_outliers, 2)*2 - torch.tensor([[10.0, 10.0]])
indices = torch.randperm(N)[:num_outliers]
Y_train[indices] = outliers

# Y_train = ( Y_train - torch.mean(Y_train) ) / torch.std(Y_train)


X_calibration = X_train
Y_calibration = Y_train



tau = 0.70        # Cible de couverture : tau%
batch_size = 250
num_epochs = 1000
lr = 5e-4


tau_param = TauParameterAnnealer(tau,                
                 warm_start_step=500, 
                 tau_low_target_step=100, 
                 tau_low_steepness=1e-3,
                 tau_high_target_step=100, 
                 tau_high_steepness=1e-2,
                 low_error_init=0.5,   
                 low_error_max=0.01,    
                 high_error_init=0.2,  
                 high_error_max=0.01,  
                 eps=1e-5              
                 )

model = UnifiedConditionalEstimator(dim_X=dim_X, dim_y=dim_y, 
                                    cov_mode="full_cholesky", num_flow_layers=0, K=1,
                                    det_normalized=True
                                    )


model.fit(X_train, Y_train, 
          tau=tau, 
          epochs=num_epochs, 
          lr=lr, 
          batch_size=batch_size, 
          return_best=True, 
          print_every=10,
          tau_parameterAnnealer=tau_param,
          loss_function="log_volume"
          )


model.conformalize(X_calibration, Y_calibration, tau, fake_for_trial=False)
vol = model.compute_average_volume(X_train[[0]])
print(f"Volume de la région de confiance: {vol.item():.4f}")

# plot_1X(model, tau, X_train, Y_train)

standardized_residuals = StandardizedResiduals(dim_X, 
                                                dim_y
                                                )

standardized_residuals.fit(X_train=X_train, 
                    y_train=Y_train,
                    num_epochs=num_epochs,
                    batch_size=batch_size,
                    lr=lr,
                    verbose = 1
                    )

standardized_residuals.conformalize(x=X_calibration, y=Y_calibration, alpha = 1-tau)
volumes  = standardized_residuals.get_average_volume(X_train[0])

print("Average Volume:", volumes)

plot_with_gaussian(model,
 standardized_residuals,
 tau,
  X_train,
   Y_train,
    res_points=1000,
     fontsize=20,
    #   name="ouliers_V1",
    name=None,
       size_points=20,
        color_set="#FF5400",
         color_points="#FCFCFC",
          color_bound_points="#46237A"
          )

In [ ]:
plot_with_gaussian(model,
 standardized_residuals,
 tau,
  X_train,
   Y_train,
    res_points=1000,
     fontsize=20,
      name="ouliers_V1",
       size_points=20,
      #  color_gaussian = ""
        color_set="#FF5400",
         color_points="#FCFCFC",
          color_bound_points="#46237A"
          )

plot_with_gaussian(model,
 standardized_residuals,
 tau,
  X_train,
   Y_train,
    res_points=1000,
     fontsize=20,
      alpha_contour=0.1,
       size_points=10,
        name="ouliers_V2",
      #   color_gaussian = ""
         color_set="#0C7C59",
          color_points="#BAC1B8"
          , color_bound_points="#58A4B0"
          )


In [ ]:
torch.manual_seed(42)

N = 2000
dim_X = 1
dim_y = 2

X_train = torch.ones(N, dim_X)
Y_train = torch.empty(N, dim_y).exponential_(1.0)

X_calibration = torch.ones(N, dim_X)
Y_calibration = torch.empty(N, dim_y).exponential_(1.0)

tau = 0.3   
batch_size = 200
num_epochs = 1000
lr = 5e-4

tau_param = TauParameterAnnealer(tau,                
                 warm_start_step=500, 
                 tau_low_target_step=100, 
                 tau_low_steepness=1e-3,
                 tau_high_target_step=100, 
                 tau_high_steepness=1e-2,
                 low_error_init=0.5,   
                 low_error_max=0.01,    
                 high_error_init=0.2,  
                 high_error_max=0.01,  
                 eps=1e-5              
                 )

model = UnifiedConditionalEstimator(dim_X=dim_X, dim_y=dim_y, 
                                    cov_mode="full_cholesky", num_flow_layers=0, K=1,
                                    det_normalized=True
                                    )


model.fit(X_train, Y_train, 
          tau=tau, 
          epochs=num_epochs, 
          lr=lr, 
          batch_size=batch_size, 
          return_best=True, 
          print_every=10,
          tau_parameterAnnealer=tau_param,
          loss_function="log_volume"
          )

model.conformalize(X_calibration, Y_calibration, tau, fake_for_trial=False)
vol = model.compute_average_volume(X_train[[0]])
print(f"Volume de la région de confiance: {vol.item():.4f}")

plot_1X(model, tau, X_train, Y_train)

standardized_residuals = StandardizedResiduals(dim_X, 
                                                dim_y
                                                )

standardized_residuals.fit(X_train=X_train, 
                    y_train=Y_train,
                    num_epochs=num_epochs,
                    batch_size=batch_size,
                    lr=lr,
                    verbose = 1
                    )

standardized_residuals.conformalize(x=X_calibration, y=Y_calibration, alpha = 1-tau)
volumes  = standardized_residuals.get_average_volume(X_train[0])

print("Average Volume:", volumes)

In [ ]:
plot_with_gaussian(model,
 standardized_residuals,
 tau,
  X_train,
   Y_train,
    res_points=1000,
     fontsize=20,
      name="2D_exponential_V1",
       size_points=20,
      #  color_gaussian = ""
        color_set="#FF5400",
         color_points="#FCFCFC",
          color_bound_points="#46237A"
          )

plot_with_gaussian(model,
 standardized_residuals,
 tau,
  X_train,
   Y_train,
    res_points=1000,
     fontsize=20,
      alpha_contour=0.1,
       size_points=10,
        name="2D_exponential_V2",
      #   color_gaussian = ""
         color_set="#0C7C59",
          color_points="#BAC1B8"
          , color_bound_points="#58A4B0"
          )


In [ ]:
import numpy as np



torch.manual_seed(42)

N = 4000
dim_X = 1
dim_y = 2

# X est constant (identique pour tous)
X_train = torch.ones(N, dim_X)

# Génération de samples 2D (X et Y indépendants)
# np.random.standard_cauchy génère des échantillons avec x0=0 et gamma=1
x = np.random.standard_cauchy(N)
y = np.random.standard_cauchy(N)

Y_train = torch.tensor(np.column_stack((x, y)) , dtype=torch.float32)



X_calibration = X_train
Y_calibration = Y_train



tau = 0.70        # Cible de couverture : tau%
batch_size = 250
num_epochs = 1_000
lr = 5e-4


tau_param = TauParameterAnnealer(tau,                
                 warm_start_step=500, 
                 tau_low_target_step=100, 
                 tau_low_steepness=1e-3,
                 tau_high_target_step=100, 
                 tau_high_steepness=1e-2,
                 low_error_init=0.5,   
                 low_error_max=0.01,    
                 high_error_init=0.2,  
                 high_error_max=0.01,  
                 eps=1e-5              
                 )

model = UnifiedConditionalEstimator(dim_X=dim_X, dim_y=dim_y, 
                                    cov_mode="full_cholesky", num_flow_layers=0, K=1,
                                    det_normalized=True
                                    )


model.fit(X_train, Y_train, 
          tau=tau, 
          epochs=num_epochs, 
          lr=lr, 
          batch_size=batch_size, 
          return_best=True, 
          print_every=10,
          tau_parameterAnnealer=tau_param,
          loss_function="log_volume"
          )


model.conformalize(X_calibration, Y_calibration, tau, fake_for_trial=False)
vol = model.compute_average_volume(X_train[[0]])
print(f"Volume de la région de confiance: {vol.item():.4f}")

# plot_1X(model, tau, X_train, Y_train)

standardized_residuals = StandardizedResiduals(dim_X, 
                                                dim_y
                                                )

standardized_residuals.fit(X_train=X_train, 
                    y_train=Y_train,
                    num_epochs=num_epochs,
                    batch_size=batch_size,
                    lr=lr,
                    verbose = 1
                    )

standardized_residuals.conformalize(x=X_calibration, y=Y_calibration, alpha = 1-tau)
volumes  = standardized_residuals.get_average_volume(X_train[0])

print("Average Volume:", volumes)

plot_with_gaussian(model,
 standardized_residuals,
 tau,
  X_train,
   Y_train,
    res_points=1000,
     fontsize=20,
    #   name="ouliers_V1",
    name=None,
       size_points=20,
        color_set="#FF5400",
         color_points="#FCFCFC",
          color_bound_points="#46237A",
          ylim=(-8, 8),
          xlim=(-8, 8),
          
          )